# Elaborative Rehearsal (B) — Data Prep (stage 1)

- **input** = the source article, truncated to `configs/chunking.yaml`'s
  `max_words` (350 words), matching the chunk scale B actually sees at
  inference time.
- **target** = XSum's one-sentence summary — *abstractive* by construction.

This is the mirror of `04_rehearsal_maintenance_prep.ipynb`, with the one
difference that defines the whole A/B contrast: A's target is sentences
lifted verbatim from the input, B's target is a sentence that mostly does
**not** appear in the input.

## Why XSum rather than cnn_dailymail

`cnn_dailymail` highlights are near-extractive — enough of their n-grams
appear in the article that a model trained on them learns to copy, which is
exactly what A wants and exactly what B must not do. XSum (Narayan et al.,
EMNLP 2018, "Don't Give Me the Details, Just the Summary!") was built for
*extreme* summarization: one sentence, written to require abstraction rather
than extraction. Training B on it is what makes the manipulation check in
`07` (novel n-gram ratio should be **high** for B, near zero for A) mean
anything.

## Where this sits in the two-stage plan

Stage 1 (this notebook + `07`) teaches the abstractive compression *skill* on
single documents. It cannot teach cross-chunk integration, because
`(document → summary)` pairs contain no notion of a preceding chunk — a
limitation flagged in `03_실험 설계` §정교화 되뇌기(B) 구현 설계.

Stage 2 (not yet built) is the rolling curation loop: `(S_{i-1}, C_i) → S_i`
with a teacher, self-generated probes, a corrective retry, and selective
augmentation against a verbatim fallback.

Its "QG teacher" is a large LLM (`llama-3.1-8b-instruct`), **not** the
fine-tuned QG model from `09` — so stage 2 does not wait on `09`. The
dependency runs the other way if anything: the probe QA pairs stage 2
produces are reusable as *additional* training data for the testing-effect
model, per the "부산물" note in `03_실험 설계` §정교화 되뇌기(B) 구현 설계.

## Prior work this follows

| Choice here | Source |
|---|---|
| Abstractive compressor as a small seq2seq trained for downstream usefulness | RECOMP (Xu, Shi & Choi, ICLR 2024) §3.2 — T5-large, already summarization-pretrained, distilled from a larger teacher |
| Compressing to a *gist* rather than a truncation | ReadAgent (Lee et al., 2024) — gist memory over long context |
| `(previous summary + new context) → updated summary` for stage 2 | Recursively Summarizing Enables Long-Term Dialogue Memory (arXiv:2308.15022) |
| "Revise, don't append" in the stage-2 teacher prompt | C-DIC, Context-Driven Incremental Compression (ICML 2026) — the revise step only; its retrieve step is query-aware and would break this project's query-agnostic rule |

Section-level notes for each are in `03_실험 설계`; per-paper notes live in
`02_STUDY/논문리딩/`.

In [1]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()

import datasets
import pandas as pd
import yaml
from datasets import load_dataset
from transformers import AutoTokenizer

from src.pipeline.rehearsal import novel_ngram_ratio

project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env loaded: success ✅
NVIDIA_NIM_API_KEY: set ✅


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load `EdinburghNLP/xsum`

Ships `document`/`summary`/`id` and official `train`/`validation`/`test`
splits, so the same three-way split discipline as
`04_rehearsal_maintenance_prep.ipynb` applies: `validation` drives checkpoint
selection during training, `test` is touched exactly once at the end of `07`.
Reporting the validation number as the final result would be circular — the
checkpoint was chosen because it scored well there.

In [2]:
MAX_TRAIN_EXAMPLES = 3000
MAX_VAL_EXAMPLES = 300
MAX_TEST_EXAMPLES = 300

train_raw = load_dataset("EdinburghNLP/xsum", split=f"train[:{MAX_TRAIN_EXAMPLES}]")
val_raw = load_dataset("EdinburghNLP/xsum", split=f"validation[:{MAX_VAL_EXAMPLES}]")
test_raw = load_dataset("EdinburghNLP/xsum", split=f"test[:{MAX_TEST_EXAMPLES}]")
print(f"train: {len(train_raw)}, validation: {len(val_raw)}, test: {len(test_raw)}")

sample = train_raw[0]
print("\nsample keys:", list(sample.keys()))
print("document (first 300 chars):", sample["document"][:300])
print("\nsummary:", sample["summary"])

train: 3000, validation: 300, test: 300

sample keys: ['document', 'summary', 'id']
document (first 300 chars): The full cost of damage in Newton Stewart, one of the areas worst affected, is still being assessed.
Repair work is ongoing in Hawick and many roads in Peeblesshire remain badly affected by standing water.
Trains on the west coast mainline face disruption due to damage at the Lamington Viaduct.
Many

summary: Clean-up operations are continuing across the Scottish Borders and Dumfries and Galloway after flooding caused by Storm Frank.


## 2. Build (input, target) pairs

No oracle extraction step, unlike `04`. XSum already provides the target
directly, and it is abstractive by design — there is nothing to approximate.

Documents are truncated to `max_words` (350) for the same reason as `04`:
`rehearse_elaborative` will operate on a single chunk at inference, so the
training input has to be the same scale. Unlike `04` the truncation cannot
invalidate the target (the target is not required to appear in the input),
so no post-truncation validity check is needed — but truncation *can* remove
the facts the summary refers to, which would train the model to invent them.
XSum documents are BBC articles and front-loaded like news generally is, so
we measure how much of the summary's vocabulary survives truncation rather
than assuming it.

In [3]:
CHUNK_MAX_WORDS = yaml.safe_load(open("configs/chunking.yaml", encoding="utf-8"))["max_words"]


def truncate_words(text: str, max_words: int) -> str:
    return " ".join(text.split()[:max_words])


def build_pair(example: dict) -> dict | None:
    truncated = truncate_words(example["document"], CHUNK_MAX_WORDS)
    summary = example["summary"].strip()
    if not truncated.strip() or not summary:
        return None
    return {"id": example["id"], "input_text": truncated, "target_text": summary}


train_pairs = [p for p in (build_pair(ex) for ex in train_raw) if p is not None]
val_pairs = [p for p in (build_pair(ex) for ex in val_raw) if p is not None]
test_pairs = [p for p in (build_pair(ex) for ex in test_raw) if p is not None]
print(f"train pairs: {len(train_pairs)} / {len(train_raw)}")
print(f"val pairs:   {len(val_pairs)} / {len(val_raw)}")
print(f"test pairs:  {len(test_pairs)} / {len(test_raw)}")

pd.DataFrame(train_pairs)[["input_text", "target_text"]].head(3)

train pairs: 3000 / 3000
val pairs:   299 / 300
test pairs:  300 / 300


,input_text,target_text
0,"The full cost of damage in Newton Stewart, one...",Clean-up operations are continuing across the ...
1,A fire alarm went off at the Holiday Inn in Ho...,Two tourist buses have been destroyed by fire ...
2,Ferrari appeared in a position to challenge un...,Lewis Hamilton stormed to pole position at the...


### Truncation sanity check + the abstractiveness that motivates B

Two things measured on the same sample:

1. **Content survival** — how much of the summary's content vocabulary is
   still present after cutting to 350 words. If this were low, B would be
   trained to hallucinate.
2. **Novel n-gram ratio of the target itself** — the fraction of the
   summary's trigrams absent from the (truncated) document. This is the same
   statistic `07` uses as B's manipulation check, measured here on the
   *reference* summaries to establish what "properly abstractive" looks like.
   Compare against `04`'s oracle targets, which are extracted from the input
   and therefore sit at 0.0 by construction.

In [4]:
import re

_STOP = set(
    "a an the and or but if of to in on at for with by from as is are was were be been it its this that "
    "he she they them his her their has have had will would can could not no".split()
)


def content_overlap(summary: str, document: str) -> float:
    """Fraction of the summary's content words that appear in the document."""
    words = [w for w in re.findall(r"[a-z']+", summary.lower()) if w not in _STOP]
    if not words:
        return 1.0
    doc = set(re.findall(r"[a-z']+", document.lower()))
    return sum(w in doc for w in words) / len(words)


sample_pairs = train_pairs[:300]
overlaps = [content_overlap(p["target_text"], p["input_text"]) for p in sample_pairs]
novelty = [novel_ngram_ratio(p["target_text"], p["input_text"], n=3) for p in sample_pairs]

print(f"on {len(sample_pairs)} training pairs (after 350-word truncation):")
print(f"  summary content words present in input : mean {sum(overlaps) / len(overlaps):.3f}")
print(f"  target novel 3-gram ratio              : mean {sum(novelty) / len(novelty):.3f}")
print("\n(A's oracle targets are extracted from the input, so their novel 3-gram")
print(" ratio is 0.0 by construction. B's targets should be far above that —")
print(" that gap is the manipulation the experiment claims to make.)")

on 300 training pairs (after 350-word truncation):
  summary content words present in input : mean 0.427
  target novel 3-gram ratio              : mean 0.980

(A's oracle targets are extracted from the input, so their novel 3-gram
 ratio is 0.0 by construction. B's targets should be far above that —
 that gap is the manipulation the experiment claims to make.)


## 3. Tokenize

`INPUT_MAX_LENGTH` matches `04` (512 covers a 350-word English document to
roughly the 90th percentile with the `t5-small` tokenizer at ~1.43
tokens/word).

`TARGET_MAX_LENGTH` is much smaller than `04`'s 160: XSum summaries are a
single sentence, where A's oracle targets ran up to three. Measured below
rather than assumed — re-measure if the backbone or `max_words` changes.

In [5]:
INPUT_MAX_LENGTH = 512
TARGET_MAX_LENGTH = 64

tokenizer = AutoTokenizer.from_pretrained("t5-small")

for name, pairs in [("input", "input_text"), ("target", "target_text")]:
    lengths = sorted(len(tokenizer(p[pairs]).input_ids) for p in train_pairs)
    pct = lambda q: lengths[int(q * (len(lengths) - 1))]  # noqa: E731
    print(f"{name:7} token length  50th={pct(0.50):>4}  90th={pct(0.90):>4}  99th={pct(0.99):>4}")


def tokenize_pairs(pairs: list[dict]) -> datasets.Dataset:
    inputs = tokenizer([p["input_text"] for p in pairs], max_length=INPUT_MAX_LENGTH, truncation=True)
    targets = tokenizer([p["target_text"] for p in pairs], max_length=TARGET_MAX_LENGTH, truncation=True)
    return datasets.Dataset.from_dict(
        {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": targets["input_ids"],
        }
    )


train_dataset = tokenize_pairs(train_pairs)
val_dataset = tokenize_pairs(val_pairs)
test_dataset = tokenize_pairs(test_pairs)
print()
print(train_dataset)

Token indices sequence length is longer than the specified maximum sequence length for this model (545 > 512). Running this sequence through the model will result in indexing errors


input   token length  50th= 268  90th= 297  99th= 338
target  token length  50th=  30  90th=  39  99th=  57

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3000
})


## 4. Save

`07_rehearsal_elaborative_train.ipynb` reads these directly. The raw text
pairs are kept alongside because `07`'s manipulation check needs the
untokenized source to compare generations against.

In [6]:
OUT_DIR = Path("data/processed/rehearsal_elaborative")
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_dataset.save_to_disk(str(OUT_DIR / "train"))
val_dataset.save_to_disk(str(OUT_DIR / "val"))
test_dataset.save_to_disk(str(OUT_DIR / "test"))

pd.DataFrame(train_pairs).to_csv(OUT_DIR / "train_pairs_raw.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(val_pairs).to_csv(OUT_DIR / "val_pairs_raw.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(test_pairs).to_csv(OUT_DIR / "test_pairs_raw.csv", index=False, encoding="utf-8-sig")

print(f"Saved to: {OUT_DIR}")
for name, ds in [("train", train_dataset), ("val", val_dataset), ("test", test_dataset)]:
    print(f"  {name}: {len(ds)} rows")

Saving the dataset (1/1 shards): 100%|██████████| 300/300 [00:00<00:00, 89788.15 examples/s] 

Saved to: data/processed/rehearsal_elaborative
  train: 3000 rows
  val: 299 rows
  test: 300 rows


## Summary

- Source: `EdinburghNLP/xsum`, official three-way split, sliced to
  3,000/300/300 for this run. `test` is touched once, at the end of `07`.
- Documents truncated to `configs/chunking.yaml`'s `max_words` (350) to match
  the chunk scale B sees at inference.
- **No oracle extraction step** — unlike `04`, the target is given and is
  abstractive by construction. That is the entire point: A learns to keep the
  source's own sentences, B learns to rewrite them.
- Next: `07_rehearsal_elaborative_train.ipynb`. Its manipulation check is the
  mirror image of `05`'s — B's novel n-gram ratio should be **high**, and a
  B that scores near zero has silently become A.
- Stage 2 (rolling gisting curation with a teacher, self-probes, corrective
  retry, and selective augmentation against a verbatim fallback) is specified
  in `03_실험 설계` §정교화 되뇌기(B) 구현 설계 and is blocked on the QG
  model from `09`.